In [ ]:
import geopandas as gpd
import pandas as pd
lia = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/Himalaya_LIA_with_RGIID_Complete/Himalaya_LIA_with_RGIID_Complete.shp")
rgi = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/himalaya_glaciers_complete/himalaya_glaciers_complete.shp")

In [ ]:
# 筛选 RGI 面积 >= 10 km²
rgi_large = rgi[rgi['Area'] >= 10].copy()
large_rgi_ids = set(rgi_large['RGIId'])

# 找包含这些大冰川的 LIA
def contains_large_rgi(rgi_str):
    if pd.isna(rgi_str):
        return False
    ids = [x.strip() for x in str(rgi_str).split(';')]
    return bool(set(ids) & large_rgi_ids)

lia_matched = lia[lia['AllRGIs'].apply(contains_large_rgi)].copy()

print("LIA筛选完成: {} 条".format(len(lia_matched)))

In [ ]:
# 筛选MainRGI对应的RGI冰川
main_rgi_ids = lia_matched['MainRGI'].tolist()
rgi_to_calibrate = rgi[rgi['RGIId'].isin(main_rgi_ids)].copy()

# 转WGS84
lia_matched = lia_matched.to_crs("EPSG:4326")
rgi_to_calibrate = rgi_to_calibrate.to_crs("EPSG:4326")

# 建立映射字典: LIA的UniqueID → MainRGI
lia_to_main_rgi = dict(zip(
    lia_matched['UniqueID'].astype(int),
    lia_matched['MainRGI']
))

print("RGI待校准: {} 条".format(len(rgi_to_calibrate)))
print("映射关系: {} 条".format(len(lia_to_main_rgi)))

In [ ]:
from oggm import cfg, workflow, tasks, utils

WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_rgi_calibration"
DEM_PATH = "/mnt/d/01 文件整理/04 数据/HMA_newregion/DEM_clip/Himalaya_3regions_DEM.tif"
MB_PATHS = [
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_13/dh_13_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_14/dh_14_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_15/dh_15_rgi60_pergla_rates.csv",
]
MB_PERIOD = "2000-01-01_2020-01-01"
# 读取MB数据
dfs = []
for path in MB_PATHS:
    print("读取: {}".format(path))
    df = pd.read_csv(path)
    dfs.append(df)
mb_all = pd.concat(dfs, ignore_index=True)
mb_data = mb_all[mb_all['period'] == MB_PERIOD].copy()
mb_data['dmdtda_kg'] = mb_data['dmdtda'] * 1000
mb_data = mb_data.set_index('rgiid')
print("MB数据: {} 条".format(len(mb_data)))
# 清理RGI数据
rgi = rgi_to_calibrate.copy()
rgi = rgi[rgi.geometry.notna() & ~rgi.is_empty].copy()
rgi['geometry'] = rgi.geometry.buffer(0)
rgi = rgi[rgi.is_valid].copy()
rgi = rgi[rgi['RGIId'].isin(mb_data.index)].copy()
print("有MB数据的RGI冰川: {}".format(len(rgi)))

In [ ]:
from oggm import cfg, workflow, tasks, GlacierDirectory
cfg.initialize()

cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

gdirs_rgi = workflow.init_glacier_directories(rgi, reset=False)
print("初始化完成: {}".format(len(gdirs_rgi)))

In [ ]:
for gdir in gdirs_rgi:
    tasks.define_glacier_region(gdir, source='USER')
print("define_glacier_region完成: {}".format(len(gdirs_rgi)))

In [ ]:
cfg.PARAMS['baseline_climate'] = 'CRU'
for gdir in gdirs_rgi:
    tasks.process_cru_data(gdir)
print("气候数据处理完成")

In [ ]:
from oggm.workflow import execute_entity_task
cfg.PARAMS['use_multiprocessing'] = True

execute_entity_task(tasks.glacier_masks, gdirs_rgi)
execute_entity_task(tasks.compute_centerlines, gdirs_rgi)
execute_entity_task(tasks.initialize_flowlines, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_line, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs_rgi)
execute_entity_task(tasks.catchment_area, gdirs_rgi)
execute_entity_task(tasks.catchment_intersections, gdirs_rgi)
execute_entity_task(tasks.catchment_width_geom, gdirs_rgi)
execute_entity_task(tasks.catchment_width_correction, gdirs_rgi)
print("预处理完成")

In [ ]:
cfg.PARAMS['use_winter_prcp_fac'] = False
cfg.PARAMS['prcp_fac'] = 8
from oggm.core.massbalance import mb_calibration_from_scalar_mb
import os
import json
success = 0
fail = 0
for gdir in gdirs_rgi:
    try:
        ref_mb = float(mb_data.loc[gdir.rgi_id, 'dmdtda_kg'])
        mb_calibration_from_scalar_mb(
            gdir,
            ref_mb=ref_mb,
            ref_mb_years=(2000, 2020),
            overwrite_gdir=True,
            calibrate_param1='melt_f',
            calibrate_param2='temp_bias',
        )
        success += 1
    except Exception as e:
        fail += 1
        if fail <= 3:
            print("失败: {} - {}".format(gdir.rgi_id, str(e)[:100]))

print("\nCRU校准完成: 成功 {}, 失败 {}".format(success, fail))

# 验证几条
for gdir in gdirs_rgi[:5]:
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        p = json.load(f)
    print("{}: melt_f={:.3f} temp_bias={:.3f} baseline={}".format(
        gdir.rgi_id, p['melt_f'], p['temp_bias'], p['baseline_climate_source']))

In [ ]:
lia_sorted = lia_matched.sort_values('UniqueID').copy()
lia_cooked = utils.cook_rgidf(
    lia_sorted,
    o1_region='15',
    bgndate='16500101',
    ids=lia_sorted['UniqueID'].astype(int).tolist(),
    assign_column_values={'LIAArea': 'Area'}
)

lia_cooked['UniqueID'] = lia_sorted['UniqueID'].values
lia_cooked['MainRGI'] = lia_sorted['MainRGI'].values

print("cook完成: {} 条".format(len(lia_cooked)))
print(lia_cooked[['RGIId', 'UniqueID', 'Area']].head())

In [ ]:
LIA_WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_oggm_LIA_spinup"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"


cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

gdirs_lia = workflow.init_glacier_directories(lia_cooked,reset= False)
print("LIA初始化完成: {} 条".format(len(gdirs_lia)))

In [ ]:
# 1. 重建lia_mb_params（用新的CRU校准参数）
import json, os, numpy as np

rgi_params_cru = {}
for gdir in gdirs_rgi:
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        rgi_params_cru[gdir.rgi_id] = json.load(f)

lia_mb_params_cru = {}
missing = []
for idx, row in lia_matched.iterrows():
    unique_id = int(row['UniqueID'])
    main_rgi = row['MainRGI']
    if main_rgi in rgi_params_cru:
        p = rgi_params_cru[main_rgi]
        lia_mb_params_cru[unique_id] = {
            'melt_f': p['melt_f'],
            'prcp_fac': p['prcp_fac'],
            'temp_bias': p['temp_bias'],
        }
    else:
        missing.append((unique_id, main_rgi))

# 缺失的用区域平均
avg_melt_f = np.mean([p['melt_f'] for p in rgi_params_cru.values()])
avg_temp_bias = np.mean([p['temp_bias'] for p in rgi_params_cru.values()])
for uid, rgi in missing:
    lia_mb_params_cru[uid] = {'melt_f': avg_melt_f, 'prcp_fac': 8.0, 'temp_bias': avg_temp_bias}

print("LIA参数映射: {} 条, 缺失补充: {} 条".format(len(lia_mb_params_cru), len(missing)))
print("melt_f范围: {:.2f} ~ {:.2f}".format(
    min(p['melt_f'] for p in lia_mb_params_cru.values()),
    max(p['melt_f'] for p in lia_mb_params_cru.values())))

# 2. 写入LIA冰川的mb_calib.json
for gdir in gdirs_lia:
    unique_id = int(gdir.rgi_id.split('.')[-1])
    params = lia_mb_params_cru[unique_id]

    mb_calib = {
        'rgi_id': gdir.rgi_id,
        'bias': 0,
        'melt_f': params['melt_f'],
        'prcp_fac': params['prcp_fac'],
        'temp_bias': params['temp_bias'],
        'reference_mb': 0,
        'reference_mb_err': None,
        'reference_period': '2000-01-01_2020-01-01',
        'mb_global_params': {
            'temp_default_gradient': -0.0065,
            'temp_all_solid': 0.0,
            'temp_all_liq': 2.0,
            'temp_melt': -1.0,
        },
        'baseline_climate_source': 'CRU TS4.04',
    }

    fpath = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fpath, 'w') as f:
        json.dump(mb_calib, f, indent=2)

print("LIA冰川mb_calib写入完成: {} 条".format(len(gdirs_lia)))

# 验证IK
with open(os.path.join(gdirs_lia[0].dir, 'mb_calib.json'), 'r') as f:
    p = json.load(f)
print("样本: melt_f={:.3f} baseline={}".format(p['melt_f'], p['baseline_climate_source']))

In [ ]:
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task
from oggm import tasks
import json, os

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

offsets = [0, -0.5, -1, -1.5, -2, -2.5, -3, -3.5, -4,-4.5,-5]

for offset in offsets:
    print("Running offset={} ...".format(offset))

    for gdir in gdirs_lia:
        uid = int(gdir.rgi_id.split('.')[-1])
        original_tb = lia_mb_params_cru[uid]['temp_bias']
        with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
            p = json.load(f)
        p['temp_bias'] = original_tb + offset
        with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
            json.dump(p, f, indent=2)

    suffix = '_offset{}'.format(str(offset).replace('.', '_').replace('-', 'n'))

    execute_entity_task(
        tasks.run_constant_climate,
        gdirs_lia,
        nyears=500,
        y0=1650,
        halfsize=30,
        climate_filename='gcm_data',
        climate_input_filesuffix='_ensemble_rec_ref',
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=suffix,
    )
    print("  offset={} 完成".format(offset))

# 恢复原始temp_bias
for gdir in gdirs_lia:
    uid = int(gdir.rgi_id.split('.')[-1])
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        p = json.load(f)
    p['temp_bias'] = lia_mb_params_cru[uid]['temp_bias']
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
        json.dump(p, f, indent=2)

print("\n所有offset实验完成，temp_bias已恢复")

In [ ]:
import xarray as xr
import numpy as np
from scipy.interpolate import interp1d

offsets = [0, -0.5, -1, -1.5, -2, -2.5, -3, -3.5, -4, -4.5, -5]
results = {}

for gdir in gdirs_lia:
    rgi_id = gdir.rgi_id
    area_changes = {}

    for offset in offsets:
        suffix = '_offset{}'.format(str(offset).replace('.', '_').replace('-', 'n'))
        try:
            f = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
            ds = xr.open_dataset(f)
            area_init = float(ds['area_m2'].values[0])
            area_end = float(ds['area_m2'].values[-1])
            pct = (area_end - area_init) / area_init * 100
            ds.close()
            area_changes[offset] = pct
        except:
            pass

    if len(area_changes) < 2:
        results[rgi_id] = {'best_offset': np.nan, 'note': 'insufficient_data'}
        continue

    offs = sorted(area_changes.keys())
    pcts = [area_changes[o] for o in offs]

    # 找零点crossing
    best_offset = np.nan
    for i in range(len(pcts) - 1):
        if pcts[i] * pcts[i+1] <= 0:  # 跨零
            # 线性插值
            f_interp = interp1d([pcts[i], pcts[i+1]], [offs[i], offs[i+1]])
            best_offset = float(f_interp(0))
            break

    if np.isnan(best_offset):
        if all(p < 0 for p in pcts):
            note = 'all_shrink: 0={:.1f}%, -5={:.1f}%'.format(pcts[0], pcts[-1])
        elif all(p > 0 for p in pcts):
            note = 'all_grow: 0={:.1f}%, -5={:.1f}%'.format(pcts[0], pcts[-1])
        else:
            note = 'no_crossing'
        results[rgi_id] = {'best_offset': np.nan, 'note': note}
    else:
        results[rgi_id] = {'best_offset': best_offset, 'note': 'success'}

# 统计
successful = [r for r in results.values() if not np.isnan(r['best_offset'])]
failed = [r for r in results.values() if np.isnan(r['best_offset'])]

print("=== Offset实验结果 ===")
print("成功: {}/{}".format(len(successful), len(results)))
print("失败: {}".format(len(failed)))

if successful:
    best_offsets = [r['best_offset'] for r in successful]
    print("\nbest_offset统计:")
    print("  mean: {:.2f}".format(np.mean(best_offsets)))
    print("  median: {:.2f}".format(np.median(best_offsets)))
    print("  std: {:.2f}".format(np.std(best_offsets)))
    print("  range: {:.2f} ~ {:.2f}".format(min(best_offsets), max(best_offsets)))

# 失败原因分布
from collections import Counter
fail_reasons = Counter(r['note'] for r in failed)
print("\n失败原因:")
for reason, count in fail_reasons.most_common():
    print("  {}: {}".format(reason, count))

In [ ]:
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task
from oggm import tasks, cfg
import json, os, numpy as np, xarray as xr
from scipy.interpolate import interp1d
import pandas as pd
import pickle

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

windows = [1300, 1350, 1400, 1450, 1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850]
offsets = [0, -1, -2, -3, -4, -5]

# ========== 第一部分：所有窗口的offset实验 ==========
for y0 in windows:
    print("\n===== 时间窗口 y0={} =====".format(y0))
    for offset in offsets:
        for gdir in gdirs_lia:
            uid = int(gdir.rgi_id.split('.')[-1])
            original_tb = lia_mb_params_cru[uid]['temp_bias']
            with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
                p = json.load(f)
            p['temp_bias'] = original_tb + offset
            with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
                json.dump(p, f, indent=2)

        suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
        execute_entity_task(
            tasks.run_constant_climate,
            gdirs_lia,
            nyears=300,
            y0=y0,
            halfsize=30,
            climate_filename='gcm_data',
            climate_input_filesuffix='_ensemble_rec_ref',
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            output_filesuffix=suffix,
        )
        print("  y0={}, offset={} 完成".format(y0, offset))

# 恢复原始temp_bias
for gdir in gdirs_lia:
    uid = int(gdir.rgi_id.split('.')[-1])
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        p = json.load(f)
    p['temp_bias'] = lia_mb_params_cru[uid]['temp_bias']
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
        json.dump(p, f, indent=2)
print("\n所有窗口offset实验完成，temp_bias已恢复")

# ========== 第二部分：提取并保存所有数据 ==========
save_dir = os.path.join(cfg.PATHS['working_dir'], 'experiment_results')
os.makedirs(save_dir, exist_ok=True)

# 2a. 保存完整时间序列（netCDF per window）
for y0 in windows:
    for offset in offsets:
        suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
        ts_data = {}
        for gdir in gdirs_lia:
            try:
                f = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
                ds = xr.open_dataset(f)
                ts_data[gdir.rgi_id] = {
                    'area_m2': ds['area_m2'].values,
                    'volume_m3': ds['volume_m3'].values,
                    'length_m': ds['length_m'].values,
                }
                n_years = len(ds['area_m2'].values)
                ds.close()
            except:
                pass

        # 转为xarray Dataset
        rgi_ids = sorted(ts_data.keys())
        if rgi_ids:
            ds_out = xr.Dataset({
                'area_m2': (['rgi_id', 'time'], [ts_data[r]['area_m2'] for r in rgi_ids]),
                'volume_m3': (['rgi_id', 'time'], [ts_data[r]['volume_m3'] for r in rgi_ids]),
                'length_m': (['rgi_id', 'time'], [ts_data[r]['length_m'] for r in rgi_ids]),
            }, coords={
                'rgi_id': rgi_ids,
                'time': np.arange(n_years),
            })
            ds_out.attrs['y0'] = y0
            ds_out.attrs['offset'] = offset
            nc_path = os.path.join(save_dir, 'ts_w{}_offset{}.nc'.format(
                y0, str(offset).replace('.', '_').replace('-', 'n')))
            ds_out.to_netcdf(nc_path)
    print("y0={} 时间序列保存完成".format(y0))

# 2b. 提取气候信息
climate_info = {}
for y0 in windows:
    sample_gdir = gdirs_lia[0]
    try:
        fpath = sample_gdir.get_filepath('gcm_data', filesuffix='_ensemble_rec_ref')
        ds_clim = xr.open_dataset(fpath)
        t_start = '{}-01'.format(y0 - 30)
        t_end = '{}-12'.format(y0 + 30)
        ds_sel = ds_clim.sel(time=slice(t_start, t_end))
        climate_info[y0] = {
            'mean_temp': float(ds_sel['temp'].mean()),
            'mean_prcp': float(ds_sel['prcp'].mean()),
        }
        ds_clim.close()
    except Exception as e:
        climate_info[y0] = {'mean_temp': np.nan, 'mean_prcp': np.nan, 'error': str(e)}
print("气候信息提取完成")

# 2c. 冰川参数和基础属性
glacier_info = {}
for gdir in gdirs_lia:
    uid = int(gdir.rgi_id.split('.')[-1])
    params = lia_mb_params_cru[uid]
    # LIA面积从初始状态取
    try:
        suffix0 = '_w1650_offset0'
        f = gdir.get_filepath('model_diagnostics', filesuffix=suffix0)
        ds = xr.open_dataset(f)
        lia_area = float(ds['area_m2'].values[0])
        lia_volume = float(ds['volume_m3'].values[0])
        lia_length = float(ds['length_m'].values[0])
        ds.close()
    except:
        lia_area = np.nan
        lia_volume = np.nan
        lia_length = np.nan

    glacier_info[gdir.rgi_id] = {
        'melt_f': params['melt_f'],
        'temp_bias_original': params['temp_bias'],
        'prcp_fac': params['prcp_fac'],
        'lia_area_m2': lia_area,
        'lia_volume_m3': lia_volume,
        'lia_length_m': lia_length,
    }

# 2d. 计算best_offset
all_results = {}
for y0 in windows:
    results = {}
    for gdir in gdirs_lia:
        rgi_id = gdir.rgi_id
        area_changes = {}
        for offset in offsets:
            suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
            try:
                f = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
                ds = xr.open_dataset(f)
                area_init = float(ds['area_m2'].values[0])
                area_end = float(ds['area_m2'].values[-1])
                pct = (area_end - area_init) / area_init * 100
                ds.close()
                area_changes[offset] = pct
            except:
                pass

        if len(area_changes) < 2:
            results[rgi_id] = {'best_offset': np.nan, 'note': 'insufficient_data'}
            continue

        offs = sorted(area_changes.keys())
        pcts = [area_changes[o] for o in offs]

        best_offset = np.nan
        for i in range(len(pcts) - 1):
            if pcts[i] * pcts[i+1] <= 0:
                f_interp = interp1d([pcts[i], pcts[i+1]], [offs[i], offs[i+1]])
                best_offset = float(f_interp(0))
                break

        if np.isnan(best_offset):
            if all(p < 0 for p in pcts):
                note = 'all_shrink'
            elif all(p > 0 for p in pcts):
                note = 'all_grow'
            else:
                note = 'no_crossing'
            results[rgi_id] = {'best_offset': np.nan, 'note': note}
        else:
            results[rgi_id] = {'best_offset': best_offset, 'note': 'success'}

    all_results[y0] = results
    successful = [r for r in results.values() if not np.isnan(r['best_offset'])]
    if successful:
        bos = [r['best_offset'] for r in successful]
        print("y0={}: 成功={}/{}, offset: mean={:.2f}, median={:.2f}".format(
            y0, len(successful), len(results), np.mean(bos), np.median(bos)))

# ========== 第三部分：汇总CSV ==========
rows = []
for y0 in windows:
    for gdir in gdirs_lia:
        rgi_id = gdir.rgi_id
        for offset in offsets:
            suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
            try:
                f = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
                ds = xr.open_dataset(f)
                row = {
                    'y0': y0,
                    'rgi_id': rgi_id,
                    'offset': offset,
                    'area_init': float(ds['area_m2'].values[0]),
                    'area_end': float(ds['area_m2'].values[-1]),
                    'area_pct': float((ds['area_m2'].values[-1] - ds['area_m2'].values[0]) / ds['area_m2'].values[0] * 100),
                    'volume_init': float(ds['volume_m3'].values[0]),
                    'volume_end': float(ds['volume_m3'].values[-1]),
                    'length_init': float(ds['length_m'].values[0]),
                    'length_end': float(ds['length_m'].values[-1]),
                }
                ds.close()
                # 加入glacier_info
                gi = glacier_info.get(rgi_id, {})
                row.update({
                    'melt_f': gi.get('melt_f'),
                    'temp_bias_original': gi.get('temp_bias_original'),
                    'prcp_fac': gi.get('prcp_fac'),
                    'lia_area_m2': gi.get('lia_area_m2'),
                    'actual_temp_bias': gi.get('temp_bias_original', 0) + offset,
                })
                # 加入best_offset
                br = all_results.get(y0, {}).get(rgi_id, {})
                row['best_offset'] = br.get('best_offset')
                row['note'] = br.get('note')
                # 加入气候信息
                ci = climate_info.get(y0, {})
                row['window_mean_temp'] = ci.get('mean_temp')
                row['window_mean_prcp'] = ci.get('mean_prcp')
                rows.append(row)
            except:
                pass

df = pd.DataFrame(rows)
csv_path = os.path.join(save_dir, 'window_offset_experiment.csv')
df.to_csv(csv_path, index=False)
print("\nCSV保存到: {}".format(csv_path))

# 保存pickle（包含all_results, glacier_info, climate_info）
pkl_path = os.path.join(save_dir, 'window_offset_experiment.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump({
        'all_results': all_results,
        'glacier_info': glacier_info,
        'climate_info': climate_info,
        'windows': windows,
        'offsets': offsets,
    }, f)
print("Pickle保存到: {}".format(pkl_path))
print("\n全部完成！")

In [ ]:
lia_sorted = lia_matched.sort_values('UniqueID').copy()
lia_cooked = utils.cook_rgidf(
    lia_sorted,
    o1_region='15',
    bgndate='16500101',
    ids=lia_sorted['UniqueID'].astype(int).tolist(),
    assign_column_values={'LIAArea': 'Area'}
)

lia_cooked['UniqueID'] = lia_sorted['UniqueID'].values
lia_cooked['MainRGI'] = lia_sorted['MainRGI'].values

print("cook完成: {} 条".format(len(lia_cooked)))
print(lia_cooked[['RGIId', 'UniqueID', 'Area']].head())

In [ ]:
LIA_WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_oggm_LIA_spinup"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"


cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

gdirs_lia = workflow.init_glacier_directories(lia_cooked,reset= False)
print("LIA初始化完成: {} 条".format(len(gdirs_lia)))

In [ ]:
for gdir in gdirs_lia:
    tasks.define_glacier_region(gdir, source='USER')
print("define_glacier_region完成: {}".format(len(gdirs_lia)))

In [ ]:
from oggm.workflow import execute_entity_task

execute_entity_task(tasks.glacier_masks, gdirs_lia)
execute_entity_task(tasks.compute_centerlines, gdirs_lia)
execute_entity_task(tasks.initialize_flowlines, gdirs_lia)
execute_entity_task(tasks.compute_downstream_line, gdirs_lia)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs_lia)
execute_entity_task(tasks.catchment_area, gdirs_lia)
execute_entity_task(tasks.catchment_intersections, gdirs_lia)
execute_entity_task(tasks.catchment_width_geom, gdirs_lia)
execute_entity_task(tasks.catchment_width_correction, gdirs_lia)
print("预处理完成: {}".format(len(gdirs_lia)))

In [ ]:
from tqdm import tqdm

os.environ['http_proxy'] = 'http://192.168.160.1:7890'
os.environ['https_proxy'] = 'http://192.168.160.1:7890'


# cfg.PARAMS['baseline_climate'] = 'CRU'
LIA_WORKING_DIR = "/mnt/f/oggm_related/oggm_LIA_simulation1"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"


cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20
#
cfg.PARAMS['baseline_climate'] = 'CRU'
for gdir in gdirs_lia:
    tasks.process_cru_data(gdir)

# cfg.PARAMS['baseline_climate'] = 'W5E5'
# for gdir in gdirs_lia:
#     tasks.process_w5e5_data(gdir)
# print("气候数据处理完成") 参考期与重建数据不一致

In [ ]:
import xarray as xr
import os
from oggm.shop import gcm_climate

# 1. 读取重建数据
fpath_climate = "/mnt/e/3spyderProject/HMA_500/Down_scale_new/out_data/oggm_climate/climate_oggm_ensemble_all_methods.nc"
ds_clim = xr.open_dataset(fpath_climate, use_cftime=True)

temp_k = ds_clim['temp'] + 273.15  # °C → K
prcp = ds_clim['prcp']

# 2. 提取最近格点
gdir = gdirs_lia[0]
temp_sel = temp_k.sel(lon=gdir.cenlon, lat=gdir.cenlat, method='nearest')
prcp_sel = prcp.sel(lon=gdir.cenlon, lat=gdir.cenlat, method='nearest')

# 3. 处理
gcm_climate.process_gcm_data(
    gdir,
    temp=temp_sel,
    prcp=prcp_sel,
    year_range=('1961', '1990'),
    source='rec_ensemble_all',
    output_filesuffix='_rec_test',
)

print("测试完成: {}".format(gdir.rgi_id))

# 4. 验证
gcm_file = os.path.join(gdir.dir, 'gcm_data_rec_test.nc')
if os.path.exists(gcm_file):
    ds_out = xr.open_dataset(gcm_file)
    print("ref_hgt: {}".format(ds_out.attrs.get('ref_hgt')))
    print("climate_source: {}".format(ds_out.attrs.get('climate_source')))
    print("temp范围: {:.1f} ~ {:.1f}".format(float(ds_out['temp'].min()), float(ds_out['temp'].max())))
    print("时间范围: {} ~ {}".format(ds_out['time'].values[0], ds_out['time'].values[-1]))
    ds_out.close()
else:
    print("输出文件不存在!")

ds_clim.close()

In [ ]:
import xarray as xr
import os
from oggm.shop import gcm_climate

ensemble_dir = "/mnt/e/3spyderProject/HMA_500/Down_scale_new/out_data/oggm_climate/Ensemble"
climate_files = sorted([f for f in os.listdir(ensemble_dir) if f.endswith('.nc')])

for ci, cfile in enumerate(climate_files):
    suffix = '_' + cfile.replace('climate_oggm_', '').replace('.nc', '')
    fpath = os.path.join(ensemble_dir, cfile)

    print("\n[{}/{}] {} -> suffix: {}".format(ci+1, len(climate_files), cfile, suffix))

    ds_clim = xr.open_dataset(fpath, use_cftime=True)
    temp_k = ds_clim['temp'] + 273.15
    prcp = ds_clim['prcp']

    success = 0
    fail = 0

    for gdir in gdirs_lia:
        try:
            temp_sel = temp_k.sel(lon=gdir.cenlon, lat=gdir.cenlat, method='nearest')
            prcp_sel = prcp.sel(lon=gdir.cenlon, lat=gdir.cenlat, method='nearest')

            gcm_climate.process_gcm_data(
                gdir,
                temp=temp_sel,
                prcp=prcp_sel,
                year_range=('1961', '1990'),
                source='rec' + suffix,
                output_filesuffix=suffix,
            )
            success += 1
        except Exception as e:
            fail += 1
            if fail <= 3:
                print("  失败: {} - {}".format(gdir.rgi_id, str(e)[:60]))

    ds_clim.close()
    print("  完成: 成功 {}, 失败 {}".format(success, fail))

print("\n全部处理完成!")

In [ ]:
#一个诊断/验证脚本，用来检查你处理好的各个气候数据集的质量和一致性，并不做任何模拟或数据处理。
# 具体做了两件事：
# 第一部分：比较不同集合方案的温度信号
# 对第一条冰川，读取6个不同后缀的气候文件，对比它们在三个时段的平均温度：
#
# 1961-1990（现代参考期）
# 1600-1700（LIA盛期）
# 1850-1900（LIA晚期）
#
# 并以CRU观测数据作为基准，计算每个重建方案相对参考期的LIA降温幅度（lia_mean - ref_mean），输出一个对比表格。目的是看看 ensemble_6、ensemble_9、ensemble_all_methods、ensemble_rec_ref 等不同集合方案的LIA信号是否合理、彼此是否一致。
# 第二部分：检查空间一致性
# 取前10条冰川，读取 ensemble_all_methods 的数据，看不同冰川之间LIA降温信号是否有空间差异，顺带检查各冰川的参考高程（ref_hgt）是否正确写入。
import xarray as xr
import numpy as np
import os

gdir = gdirs_lia[0]
print("冰川: {} (ref_hgt: {})".format(gdir.rgi_id,
    xr.open_dataset(os.path.join(gdir.dir, 'climate_historical.nc')).attrs['ref_hgt']))

# 所有后缀
suffixes = ['ensemble_6', 'ensemble_9', 'ensemble_all_methods',
            'ensemble_my_methods', 'ensemble_rec_ref', 'ensemble_rec_result']

# CRU参考
ds_cru = xr.open_dataset(os.path.join(gdir.dir, 'climate_historical.nc'))
cru_ref = float(ds_cru['temp'].sel(time=slice('1961-01-01', '1990-12-31')).mean())
cru_1700 = "N/A"  # CRU没有1700年数据
ds_cru.close()

print("\n{:<25} {:>10} {:>10} {:>10} {:>12}".format(
    "数据集", "1961-1990", "1600-1700", "1850-1900", "LIA降温"))
print("-" * 70)
print("{:<25} {:>10.2f} {:>10} {:>10} {:>12}".format(
    "CRU", cru_ref, "N/A", "N/A", "N/A"))

for suffix in suffixes:
    fpath = os.path.join(gdir.dir, 'gcm_data_{}.nc'.format(suffix))
    if not os.path.exists(fpath):
        print("{:<25} 文件不存在".format(suffix))
        continue

    ds = xr.open_dataset(fpath, use_cftime=True)

    ref_mean = float(ds['temp'].sel(time=slice('1961', '1990')).mean())
    lia_mean = float(ds['temp'].sel(time=slice('1600', '1700')).mean())
    late_lia = float(ds['temp'].sel(time=slice('1850', '1900')).mean())

    # 也看看1500年代
    early = float(ds['temp'].sel(time=slice('1500', '1600')).mean())

    print("{:<25} {:>10.2f} {:>10.2f} {:>10.2f} {:>+12.2f}".format(
        suffix, ref_mean, lia_mean, late_lia, lia_mean - ref_mean))

    ds.close()

# 再看多条冰川的LIA信号
print("\n\n=== 多条冰川LIA降温信号 (ensemble_all_methods) ===")
print("{:<18} {:>8} {:>10} {:>10} {:>12}".format(
    "RGI_ID", "ref_hgt", "1961-1990", "1600-1700", "LIA降温"))
print("-" * 62)

for gdir in gdirs_lia[:10]:
    fpath = os.path.join(gdir.dir, 'gcm_data_ensemble_all_methods.nc')
    if not os.path.exists(fpath):
        continue

    ds = xr.open_dataset(fpath, use_cftime=True)
    ref_hgt = ds.attrs.get('ref_hgt', 0)
    ref_mean = float(ds['temp'].sel(time=slice('1961', '1990')).mean())
    lia_mean = float(ds['temp'].sel(time=slice('1600', '1700')).mean())

    print("{:<18} {:>8.0f} {:>10.2f} {:>10.2f} {:>+12.2f}".format(
        gdir.rgi_id, ref_hgt, ref_mean, lia_mean, lia_mean - ref_mean))

    ds.close()

In [ ]:
import json
import os

mb_global_params = {
    'temp_default_gradient': -0.0065,
    'temp_all_solid': 0.0,
    'temp_all_liq': 2.0,
    'temp_melt': -1.0
}

success = 0
for gdir in gdirs_lia:
    try:
        unique_id = int(gdir.rgi_id.split('.')[-1])
        params = lia_mb_params[unique_id]

        mb_calib = {
            'rgi_id': gdir.rgi_id,
            'melt_f': params['melt_f'],
            'prcp_fac': params['prcp_fac'],
            'temp_bias': params['temp_bias'],
            'bias': 0,
            'reference_mb': 0,
            'reference_mb_err': None,
            'reference_period': '2000-01-01_2020-01-01',
            'mb_global_params': mb_global_params,
            'baseline_climate_source': 'CRU TS4.04',
        }

        fpath = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fpath, 'w') as f:
            json.dump(mb_calib, f, indent=2)
        success += 1
    except Exception as e:
        print("失败: {} - {}".format(gdir.rgi_id, str(e)[:60]))

print("mb_calib更新完成: {}".format(success))


# 2. 验证一下climate_historical和mb_calib的source是否匹配
import xarray as xr
gdir = gdirs_lia[0]
ds = xr.open_dataset(os.path.join(gdir.dir, 'climate_historical.nc'))
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
    mb = json.load(f)
print("\nclimate_historical source: {}".format(ds.attrs.get('climate_source')))
print("mb_calib source: {}".format(mb['baseline_climate_source']))
ds.close()

In [ ]:
# 1. mb_calib已更新，直接跑冰厚反演流程
execute_entity_task(tasks.apparent_mb_from_any_mb, gdirs_lia,
                    mb_years=(1961, 1990))
print("apparent_mb完成")

execute_entity_task(tasks.prepare_for_inversion, gdirs_lia)
execute_entity_task(tasks.mass_conservation_inversion, gdirs_lia)
execute_entity_task(tasks.filter_inversion_output, gdirs_lia)
print("冰厚反演完成")

execute_entity_task(tasks.init_present_time_glacier, gdirs_lia)
print("动力学初始化完成: {}".format(len(gdirs_lia)))

In [ ]:
from oggm.core.flowline import run_constant_climate
from oggm.core.flowline import run_constant_climate, FluxBasedModel
import json
import os
import numpy as np
import xarray as xr
import copy

t0 = 1650
climate_suffix = '_ensemble_rec_ref'

def get_area_change(gdir, offset, original_bias, mb_template):
    mb_tmp = copy.deepcopy(mb_template)
    mb_tmp['temp_bias'] = original_bias + offset
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
        json.dump(mb_tmp, f, indent=2)

    run_constant_climate(
        gdir, nyears=300, y0=t0, halfsize=30,
        climate_filename='gcm_data',
        climate_input_filesuffix=climate_suffix,
        output_filesuffix='_bisect',
        store_model_geometry=False,
        evolution_model=FluxBasedModel,
    )

    fp = gdir.get_filepath('model_diagnostics', filesuffix='_bisect')
    with xr.open_dataset(fp) as ds:
        a0 = float(ds['area_m2'].values[0])
        a_end = float(ds['area_m2'].values[-1])
    return (a_end - a0) / a0 * 100


results = {}
for i, gdir in enumerate(gdirs_lia):
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        mb = json.load(f)
    original_bias = mb['temp_bias']

    try:
        change_0 = get_area_change(gdir, 0, original_bias, mb)
        change_m4 = get_area_change(gdir, -4.0, original_bias, mb)

        # 检查是否可以二分
        if change_0 * change_m4 > 0:
            # 同号，零点不在[-4, 0]区间
            results[gdir.rgi_id] = {'best_offset': np.nan,
                                     'note': 'no_root: 0={:.1f}%, -4={:.1f}%'.format(change_0, change_m4)}
        else:
            lo, hi = -4.0, 0.0
            for _ in range(6):
                mid = (lo + hi) / 2.0
                change_mid = get_area_change(gdir, mid, original_bias, mb)
                if change_mid > 0:
                    hi = mid
                else:
                    lo = mid

            best_offset = (lo + hi) / 2.0
            results[gdir.rgi_id] = {'best_offset': best_offset}

    except Exception as e:
        results[gdir.rgi_id] = {'best_offset': np.nan, 'note': str(e)[:60]}

    # 恢复
    mb['temp_bias'] = original_bias
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
        json.dump(mb, f, indent=2)

    if (i+1) % 20 == 0:
        valid = [r['best_offset'] for r in results.values() if not np.isnan(r['best_offset'])]
        print("进度: {}/{}, 有效: {}".format(i+1, len(gdirs_lia), len(valid)))

# 统计
print("\n=== 完成 ===")
valid_offsets = [r['best_offset'] for r in results.values() if not np.isnan(r['best_offset'])]
failed = [rid for rid, r in results.items() if np.isnan(r['best_offset'])]
print("有效: {}/{}".format(len(valid_offsets), len(results)))
print("best_offset: {:.2f} ~ {:.2f}, 均值 {:.2f}".format(
    min(valid_offsets), max(valid_offsets), np.mean(valid_offsets)))
print("失败: {}".format(len(failed)))
if failed[:5]:
    for rid in failed[:5]:
        print("  {}: {}".format(rid, results[rid].get('note', '')))

execute_entity_task(tasks.prepare_for_inversion, gdirs_lia)
execute_entity_task(tasks.mass_conservation_inversion, gdirs_lia)
execute_entity_task(tasks.filter_inversion_output, gdirs_lia)

print("冰厚反演完成: {}".format(len(gdirs_lia)))

In [ ]:
# 分析失败原因
cfl_error = 0
no_root = 0
file_not_found = 0
other = 0

for rgi_id, r in results.items():
    if not np.isnan(r['best_offset']):
        continue
    note = r.get('note', '')
    if 'CFL' in note or 'No such file' in note:
        cfl_error += 1
    elif 'no_root' in note:
        no_root += 1
    else:
        other += 1

print("失败分析 (共{}):".format(len(results) - 74))
print("  CFL/文件不存在: {}".format(cfl_error))
print("  无零点(no_root): {}".format(no_root))
print("  其他: {}".format(other))

# 看看no_root的具体情况
print("\nno_root详情:")
for rgi_id, r in results.items():
    if 'no_root' in r.get('note', ''):
        print("  {}: {}".format(rgi_id, r['note']))

In [ ]:
# 对比两次校准的关键差异
import json

# 之前的校准
print("=== 之前校准 (oggm_rgi_calibration) ===")
print("melt_f: {}".format(old_rgi['melt_f']))
print("temp_bias: {}".format(old_rgi['temp_bias']))
print("prcp_fac: {}".format(old_rgi['prcp_fac']))
print("reference_mb: {}".format(old_rgi['reference_mb']))
print("baseline: {}".format(old_rgi['baseline_climate_source']))

# 当前的校准
for gdir in gdirs_rgi:
    if gdir.rgi_id == 'RGI60-15.01141':
        with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
            cur = json.load(f)
        print("\n=== 当前校准 (gdirs_rgi) ===")
        print("melt_f: {}".format(cur['melt_f']))
        print("temp_bias: {}".format(cur['temp_bias']))
        print("prcp_fac: {}".format(cur['prcp_fac']))
        print("reference_mb: {}".format(cur['reference_mb']))
        print("baseline: {}".format(cur['baseline_climate_source']))

        # 当前gdirs_rgi的climate_historical
        import xarray as xr
        ds = xr.open_dataset(os.path.join(gdir.dir, 'climate_historical.nc'))
        print("climate_historical source: {}".format(ds.attrs.get('climate_source')))
        print("climate_historical time: {} ~ {}".format(ds['time'].values[0], ds['time'].values[-1]))
        ds.close()
        break

In [ ]:
from oggm.core.flowline import run_constant_climate, FluxBasedModel
from oggm.core.massbalance import MonthlyTIModel
from functools import partial

mb_model_class = partial(MonthlyTIModel, check_calib_params=False)
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20
cfg.PARAMS['cfl_min_dt'] = 20
cfg.PARAMS['continue_on_error'] = True  #
mb_model_class = partial(MonthlyTIModel, check_calib_params=False)

workflow.execute_entity_task(
    run_constant_climate,
    gdirs_ok,
    nyears=300,
    y0=1650,
    halfsize=30,
    climate_filename='gcm_data',
    mb_model_class=mb_model_class,
    evolution_model=FluxBasedModel,
    store_model_geometry=True,
    output_filesuffix='_spinup_LIA',
)


In [ ]:
# 错误收集
import os
success_gdirs = []
failed_gdirs = []

for gdir in gdirs_ok:
    fp = gdir.get_filepath('model_diagnostics', filesuffix='_spinup_LIA')
    if os.path.exists(fp):
        success_gdirs.append(gdir.rgi_id)
    else:
        failed_gdirs.append(gdir.rgi_id)

print(f"成功: {len(success_gdirs)}, 失败: {len(failed_gdirs)}")
print(f"失败列表: {failed_gdirs}")

In [ ]:
from oggm.core.flowline import run_from_climate_data, FluxBasedModel
from oggm.workflow import execute_entity_task
from functools import partial
from oggm.core.massbalance import MonthlyTIModel

# 筛选成功spin-up的冰川
failed_spinup_ids = ['RGI60-15.00079', 'RGI60-15.00084', 'RGI60-15.00214', 'RGI60-15.01044',
                     'RGI60-15.01604', 'RGI60-15.01743', 'RGI60-15.01931', 'RGI60-15.01968',
                     'RGI60-15.02044', 'RGI60-15.02282', 'RGI60-15.02692', 'RGI60-15.02898',
                     'RGI60-15.02906', 'RGI60-15.04319', 'RGI60-15.05127', 'RGI60-15.05508']

gdirs_run = [gdir for gdir in gdirs_lia if gdir.rgi_id not in failed_spinup_ids]
print("正演冰川数量: {}".format(len(gdirs_run)))

mb_model_class = partial(MonthlyTIModel, check_calib_params=False)
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 26
cfg.PARAMS['cfl_min_dt'] = 60
cfg.PARAMS['continue_on_error'] = True

from oggm.core.massbalance import MonthlyTIModel

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

execute_entity_task(
    run_from_climate_data,
    gdirs_run,
    min_ys=1650,
    ye=2000,
    climate_filename='gcm_data',
    mb_model_class=MyMBModel,
    evolution_model=FluxBasedModel,
    init_model_filesuffix='_spinup_LIA',
    output_filesuffix='_LIA_forward'
)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

gdir = gdirs_run[150]  # 取第一个冰川

fp = gdir.get_filepath('gcm_data')
with xr.open_dataset(fp) as clim:
    # 先算年平均
    temp_annual = clim.temp.groupby('time.year').mean()

    # 10年滑动平均
    temp_10yr = temp_annual.rolling(year=30, center=True).mean()

    plt.figure(figsize=(12, 4))
    plt.plot(temp_annual.year, temp_annual.values, alpha=0.3, label='Annual')
    plt.plot(temp_10yr.year, temp_10yr.values, 'r-', linewidth=2, label='10-yr mean')
    plt.axvline(x=1450, color='g', linestyle='--', label='y0=1450')
    plt.axvline(x=1650, color='b', linestyle='--', label='min_ys=1650')
    plt.xlabel('Year')
    plt.ylabel('Temperature (K)')
    plt.title(f'LMR Temperature - {gdir.rgi_id}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 打印关键时段平均
    print("1440-1480 平均: {:.2f} K".format(float(temp_annual.sel(year=slice(1440,1480)).mean())))
    print("1650-1700 平均: {:.2f} K".format(float(temp_annual.sel(year=slice(1650,1700)).mean())))
    print("1850-1900 平均: {:.2f} K".format(float(temp_annual.sel(year=slice(1850,1900)).mean())))

In [ ]:
import json
import os
import numpy as np

temp_bias_list = []
melt_f_list = []
prcp_fac_list = []

for gdir in gdirs_run:
    try:
        fp = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fp, 'r') as f:
            params = json.load(f)

        temp_bias_list.append(params['temp_bias'])
        melt_f_list.append(params['melt_f'])
        prcp_fac_list.append(params['prcp_fac'])
    except:
        pass

print(f"统计冰川数: {len(temp_bias_list)}")
print()
print("=== temp_bias (K) ===")
print(f"  均值: {np.mean(temp_bias_list):.2f}")
print(f"  中位数: {np.median(temp_bias_list):.2f}")
print(f"  范围: {np.min(temp_bias_list):.2f} ~ {np.max(temp_bias_list):.2f}")
print(f"  标准差: {np.std(temp_bias_list):.2f}")

print()
print("=== melt_f (mm/K/day) ===")
print(f"  均值: {np.mean(melt_f_list):.2f}")
print(f"  中位数: {np.median(melt_f_list):.2f}")
print(f"  范围: {np.min(melt_f_list):.2f} ~ {np.max(melt_f_list):.2f}")
print(f"  标准差: {np.std(melt_f_list):.2f}")

print()
print("=== prcp_fac ===")
print(f"  均值: {np.mean(prcp_fac_list):.2f}")
print(f"  中位数: {np.median(prcp_fac_list):.2f}")
print(f"  范围: {np.min(prcp_fac_list):.2f} ~ {np.max(prcp_fac_list):.2f}")
print(f"  标准差: {np.std(prcp_fac_list):.2f}")

# 保存原始参数
original_tbs = {}
for gdir in gdirs_run:
    try:
        fp = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fp, 'r') as f:
            params = json.load(f)
        original_tbs[gdir.rgi_id] = params['temp_bias']
    except:
        pass

# 保存到文件
output_path = os.path.join(cfg.PATHS['working_dir'], 'original_temp_bias.json')
with open(output_path, 'w') as f:
    json.dump(original_tbs, f, indent=2)
print(f"已保存: {output_path}")
print(f"均值: {np.mean(list(original_tbs.values())):.2f}K")

In [ ]:
import json
import os
import xarray as xr
import numpy as np
from oggm.core.flowline import run_constant_climate, FluxBasedModel
from oggm.workflow import execute_entity_task
from oggm import cfg

offsets = [0,-0.5, -1,-1.5, -2,-2.5, -3, -3.5, -4]
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 22
# 设置继续运行即使出错
cfg.PARAMS['continue_on_error'] = True

# 保存原始temp_bias
original_tbs = {}
valid_gdirs = []

for gdir in gdirs_run:
    try:
        fp = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fp, 'r') as f:
            params = json.load(f)
        original_tbs[gdir.rgi_id] = params['temp_bias']
        valid_gdirs.append(gdir)
    except:
        print(f"跳过: {gdir.rgi_id}")

print(f"有效冰川: {len(valid_gdirs)}")

results = {gdir.rgi_id: {'offset_changes': {}} for gdir in valid_gdirs}

for offset in offsets:
    print(f"\n=== 测试 offset={offset:+.1f}K ===")

    # 修改所有冰川的temp_bias
    for gdir in valid_gdirs:
        fp = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fp, 'r') as f:
            params = json.load(f)
        params['temp_bias'] = original_tbs[gdir.rgi_id] + offset
        with open(fp, 'w') as f:
            json.dump(params, f)

    # 批量并行运行
    execute_entity_task(
        run_constant_climate,
        valid_gdirs,
        nyears=500,
        y0=1650,
        halfsize=30,
        climate_filename='gcm_data',
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=f'_offset{offset}'
    )

    # 收集结果
    success = 0
    for gdir in valid_gdirs:
        try:
            fp = gdir.get_filepath('model_diagnostics', filesuffix=f'_offset{offset}')
            if os.path.exists(fp):
                ds = xr.open_dataset(fp)
                a0 = ds.area_m2.values[0]
                a1 = ds.area_m2.values[-1]
                change = (a1 - a0) / a0 * 100
                ds.close()
                results[gdir.rgi_id]['offset_changes'][offset] = change
                success += 1
            else:
                results[gdir.rgi_id]['offset_changes'][offset] = None
        except:
            results[gdir.rgi_id]['offset_changes'][offset] = None

    print(f"offset={offset:+.1f}K 完成: {success}/{len(valid_gdirs)}")

# 恢复原参数
for gdir in valid_gdirs:
    fp = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fp, 'r') as f:
        params = json.load(f)
    params['temp_bias'] = original_tbs[gdir.rgi_id]
    with open(fp, 'w') as f:
        json.dump(params, f)

# 输出结果
print("\n=== 结果 ===")
print("RGI_ID\t\t\t" + "\t".join([f"{o:+.1f}K" for o in offsets]))
for rgi_id, r in list(results.items())[:30]:
    line = rgi_id
    for o in offsets:
        val = r['offset_changes'].get(o)
        line += f"\t{val:+.1f}%" if val is not None else "\tNA"
    print(line)

In [ ]:
best_offsets = {}
for rgi_id, r in results.items():
    changes = r['offset_changes']
    valid_offsets = []
    valid_changes = []
    for o in offsets:
        if changes.get(o) is not None:
            valid_offsets.append(o)
            valid_changes.append(changes[o])

    if len(valid_offsets) < 2:
        best_offsets[rgi_id] = {'best_offset': None, 'method': 'insufficient_data'}
        continue

    valid_offsets = np.array(valid_offsets)
    valid_changes = np.array(valid_changes)

    sign_changes = np.where(np.diff(np.sign(valid_changes)) != 0)[0]

    if len(sign_changes) > 0:
        idx = sign_changes[0]
        o1, o2 = valid_offsets[idx], valid_offsets[idx + 1]
        c1, c2 = valid_changes[idx], valid_changes[idx + 1]
        best_o = o1 - c1 * (o2 - o1) / (c2 - c1)
        best_offsets[rgi_id] = {'best_offset': best_o, 'method': 'interpolated'}
    else:
        min_idx = np.argmin(np.abs(valid_changes))
        best_offsets[rgi_id] = {'best_offset': valid_offsets[min_idx], 'method': 'min_abs'}

# 统计
interpolated = [r['best_offset'] for r in best_offsets.values() if r['method'] == 'interpolated' and r['best_offset'] is not None]
min_abs = [r['best_offset'] for r in best_offsets.values() if r['method'] == 'min_abs' and r['best_offset'] is not None]
insufficient = sum(1 for r in best_offsets.values() if r['method'] == 'insufficient_data')

print(f"=== 500年best_offset统计 ===")
print(f"插值找到零点: {len(interpolated)} 个冰川")
print(f"取最小绝对值: {len(min_abs)} 个冰川")
print(f"数据不足: {insufficient}")

if interpolated:
    print(f"\n插值零点统计:")
    print(f"  均值: {np.mean(interpolated):.2f}K")
    print(f"  中位数: {np.median(interpolated):.2f}K")
    print(f"  范围: {min(interpolated):.2f} ~ {max(interpolated):.2f}K")

In [ ]:
# import json
#
# # 保存到working_dir
# output_path = os.path.join(cfg.PATHS['working_dir'], 'lia_best_offsets.json')
# with open(output_path, 'w') as f:
#     json.dump(best_offsets, f, indent=2)
#
# print(f"已保存至: {output_path}")
with open(os.path.join(cfg.PATHS['working_dir'], 'lia_best_offsets.json'), 'r') as f:
    saved_offsets = json.load(f)
valid_offsets = [v['best_offset'] for v in saved_offsets.values() if v['best_offset'] is not None]
print(f"lia_best_offsets均值: {np.mean(valid_offsets):.2f}K")
print(f"lia_best_offsets中位数: {np.median(valid_offsets):.2f}K")

output_path = os.path.join(cfg.PATHS['working_dir'], 'lia_best_offsets_500yr.json')
with open(output_path, 'w') as f:
    json.dump(best_offsets, f, indent=2)
print(f"已保存: {output_path}")
###############################################################
# lia_best_offsets均值: -1.91K
# lia_best_offsets中位数: -2.01K

In [ ]:
# 保存变量
# import json
# import os
#
# working_dir = cfg.PATHS['working_dir']
#
# # 1. original_tbs 已保存
#
# # 2. best_offsets 已保存
#
# # 3. 保存valid_gdirs的rgi_id列表
# valid_ids = [gdir.rgi_id for gdir in valid_gdirs]
# with open(os.path.join(working_dir, 'valid_gdirs_ids.json'), 'w') as f:
#     json.dump(valid_ids, f)
#
# # 4. 保存results
# with open(os.path.join(working_dir, 'offset_results.json'), 'w') as f:
#     json.dump(results, f, indent=2)
#
# print("已保存:")
# print(f"  - original_temp_bias.json")
# print(f"  - lia_best_offsets.json")
# print(f"  - valid_gdirs_ids.json")
# print(f"  - offset_results.json")
#
# ############################
# 恢复变量
# with open(os.path.join(working_dir, 'original_temp_bias.json'), 'r') as f:
#     original_tbs = json.load(f)
# with open(os.path.join(working_dir, 'lia_best_offsets.json'), 'r') as f:
#     best_offsets = json.load(f)
# with open(os.path.join(working_dir, 'valid_gdirs_ids.json'), 'r') as f:
#     valid_ids = json.load(f)
#     valid_gdirs = [gdir for gdir in gdirs_run if gdir.rgi_id in valid_ids]

In [ ]:
# output_path = os.path.join(cfg.PATHS['working_dir'], 'original_temp_bias.json')
# with open(output_path, 'w') as f:
#     json.dump(original_tbs, f, indent=2)
# print(f"已保存: {output_path}")


# 3. 保存valid_gdirs的rgi_id列表
valid_ids = [gdir.rgi_id for gdir in valid_gdirs]
with open(os.path.join(working_dir, 'valid_gdirs_ids.json'), 'w') as f:
    json.dump(valid_ids, f)

# 4. 保存results
with open(os.path.join(working_dir, 'offset_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print("保存完成")

In [ ]:
best_offsets = {}
for rgi_id, r in results.items():
    changes = r['offset_changes']
    valid_offsets = []
    valid_changes = []
    for o in offsets:
        if changes.get(o) is not None:
            valid_offsets.append(o)
            valid_changes.append(changes[o])

    if len(valid_offsets) < 2:
        best_offsets[rgi_id] = {'best_offset': None, 'method': 'insufficient_data'}
        continue

    valid_offsets = np.array(valid_offsets)
    valid_changes = np.array(valid_changes)

    sign_changes = np.where(np.diff(np.sign(valid_changes)) != 0)[0]

    if len(sign_changes) > 0:
        idx = sign_changes[0]
        o1, o2 = valid_offsets[idx], valid_offsets[idx + 1]
        c1, c2 = valid_changes[idx], valid_changes[idx + 1]
        best_o = o1 - c1 * (o2 - o1) / (c2 - c1)
        best_offsets[rgi_id] = {'best_offset': best_o, 'method': 'interpolated'}
    else:
        min_idx = np.argmin(np.abs(valid_changes))
        best_offsets[rgi_id] = {'best_offset': valid_offsets[min_idx], 'method': 'min_abs'}

print(f"计算完成: {len(best_offsets)} 个冰川")

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# 加载结果
with open('all_results_t0_optimization.pkl', 'rb') as f:
    all_results = pickle.load(f)

candidate_t0s = [1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850]

# ============================================================
# 1. 统计每个t₀的best_offset分布
# ============================================================
print("=" * 60)
print("各t₀的best_offset统计")
print("=" * 60)

for t0 in candidate_t0s:
    offsets = all_results['best_offsets'][t0]
    valid = [v['best_offset'] for v in offsets.values() if v['best_offset'] is not None]
    print("t₀={}: n={}, 均值={:.2f}K, 中位数={:.2f}K, std={:.2f}K".format(
        t0, len(valid), np.mean(valid), np.median(valid), np.std(valid)))

# ============================================================
# 2. 统计每个t₀正演到2000年的冰川数量和面积变化
# ============================================================
print("\n" + "=" * 60)
print("各t₀正演结果统计")
print("=" * 60)

for t0 in candidate_t0s:
    runs = all_results['forward_runs'][t0]
    n_glaciers = len(runs)

    if n_glaciers == 0:
        print("t₀={}: 无有效结果".format(t0))
        continue

    # 计算从t₀到2000的总面积/体积变化
    area_changes = []
    vol_changes = []

    for r in runs:
        a0, a_end = r['area'][0], r['area'][-1]
        v0, v_end = r['volume'][0], r['volume'][-1]

        if a0 > 0:
            area_changes.append((a_end - a0) / a0 * 100)
        if v0 > 0:
            vol_changes.append((v_end - v0) / v0 * 100)

    print("t₀={}: n={}, 面积变化={:.1f}±{:.1f}%, 体积变化={:.1f}±{:.1f}%".format(
        t0, n_glaciers,
        np.mean(area_changes), np.std(area_changes),
        np.mean(vol_changes), np.std(vol_changes)))

# ============================================================
# 3. 与观测数据对比（如果有的话）
# ============================================================
# 假设你有LIA→现代的观测面积变化（从Lee et al. 2021）
# observed_area_change = -23.4  # 示例值

print("\n" + "=" * 60)
print("模拟vs观测对比")
print("=" * 60)

# 计算每个t₀的平均面积变化，找最接近观测值的
observed_change = -23.4  # 根据你的数据调整

t0_vs_obs = {}
for t0 in candidate_t0s:
    runs = all_results['forward_runs'][t0]
    if len(runs) == 0:
        continue

    area_changes = []
    for r in runs:
        a0, a_end = r['area'][0], r['area'][-1]
        if a0 > 0:
            area_changes.append((a_end - a0) / a0 * 100)

    mean_change = np.mean(area_changes)
    diff = abs(mean_change - observed_change)
    t0_vs_obs[t0] = {'mean_change': mean_change, 'diff_from_obs': diff}
    print("t₀={}: 模拟面积变化={:.1f}%, 与观测差={:.1f}%".format(
        t0, mean_change, diff))

# 找最优t₀
best_t0 = min(t0_vs_obs.keys(), key=lambda x: t0_vs_obs[x]['diff_from_obs'])
print("\n最优t₀ = {} (与观测最接近)".format(best_t0))

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

# 刚才图中的6条冰川
selected_ids = ['RGI60-15.04491', 'RGI60-15.02806', 'RGI60-15.04378',
                'RGI60-15.02321', 'RGI60-15.03674', 'RGI60-15.05288']

# 构建gdir索引
gdir_dict = {gdir.rgi_id: gdir for gdir in valid_gdirs_lia}

# 10年滑动平均函数
def smooth(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

colors = plt.cm.tab10(np.linspace(0, 1, len(selected_ids)))

# 存储所有冰川的气候数据
all_temp = {}
all_prcp = {}

for i, rgi_id in enumerate(selected_ids):
    gdir = gdir_dict.get(rgi_id)
    if gdir is None:
        continue

    fp = gdir.get_filepath('gcm_data')

    with xr.open_dataset(fp) as ds:
        temp = ds['temp'].values
        prcp = ds['prcp'].values
        time = ds['time'].values

        # 提取年份
        years_monthly = np.array([t.year for t in time])

        # 转为年平均
        unique_years = np.unique(years_monthly)
        temp_annual = []
        prcp_annual = []

        for yr in unique_years:
            mask = years_monthly == yr
            temp_annual.append(temp[mask].mean())
            prcp_annual.append(prcp[mask].sum())

        temp_annual = np.array(temp_annual)
        prcp_annual = np.array(prcp_annual)

        # 10年平滑
        temp_smooth = smooth(temp_annual, 10)
        prcp_smooth = smooth(prcp_annual, 10)
        years_smooth = unique_years[4:-5]

        all_temp[rgi_id] = (years_smooth, temp_smooth)
        all_prcp[rgi_id] = (years_smooth, prcp_smooth)

        # 绘制温度
        axes[0].plot(years_smooth, temp_smooth - 273.15, color=colors[i],
                     linewidth=1.5, label=rgi_id)

        # 绘制降水
        axes[1].plot(years_smooth, prcp_smooth * 12, color=colors[i],
                     linewidth=1.5, label=rgi_id)

axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('温度对比 (10年平滑)')
axes[0].axvline(1900, color='r', linestyle='--', alpha=0.5)
axes[0].legend(loc='best', fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel('Precip (mm/yr)')
axes[1].set_xlabel('Year')
axes[1].set_title('降水对比 (10年平滑)')
axes[1].axvline(1900, color='r', linestyle='--', alpha=0.5)
axes[1].legend(loc='best', fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('不同冰川LMR气候数据对比', fontsize=14)
plt.tight_layout()
plt.savefig('lmr_climate_comparison.png', dpi=150)
plt.show()

# 打印各冰川的气候差异
print("\n各冰川平均气候 (1500-2000):")
print("{:<20} {:>10} {:>12}".format("RGI_ID", "Temp(°C)", "Prcp(mm/yr)"))
print("-" * 45)
for rgi_id in selected_ids:
    if rgi_id in all_temp:
        years, temp = all_temp[rgi_id]
        _, prcp = all_prcp[rgi_id]
        print("{:<20} {:>10.2f} {:>12.1f}".format(
            rgi_id, (temp.mean() - 273.15), prcp.mean() * 12))

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

gdir_dict = {gdir.rgi_id: gdir for gdir in valid_gdirs_lia}

# 存储所有冰川的距平数据
all_anomalies = []

for gdir in valid_gdirs_lia:
    try:
        fp = gdir.get_filepath('gcm_data')

        with xr.open_dataset(fp) as ds:
            temp = ds['temp'].values
            time = ds['time'].values
            years_monthly = np.array([t.year for t in time])

            # 年平均
            unique_years = np.unique(years_monthly)
            temp_annual = []
            for yr in unique_years:
                mask = years_monthly == yr
                temp_annual.append(temp[mask].mean())
            temp_annual = np.array(temp_annual)

            # 参考期平均 (1961-1990)
            ref_mask = (unique_years >= 1961) & (unique_years <= 1990)
            t_ref = temp_annual[ref_mask].mean()

            # 各时期距平
            def get_anomaly(y1, y2):
                mask = (unique_years >= y1) & (unique_years <= y2)
                if mask.sum() > 0:
                    return temp_annual[mask].mean() - t_ref
                return np.nan

            all_anomalies.append({
                'rgi_id': gdir.rgi_id,
                'lat': gdir.cenlat,
                'lon': gdir.cenlon,
                't_ref': t_ref,
                'dt_1500': get_anomaly(1470, 1530),
                'dt_1600': get_anomaly(1570, 1630),
                'dt_1700': get_anomaly(1670, 1730),
                'dt_1750': get_anomaly(1720, 1780),
                'dt_1800': get_anomaly(1770, 1830),
                'dt_1850': get_anomaly(1820, 1880),
                'dt_1900': get_anomaly(1870, 1930),
                'dt_2000': get_anomaly(1970, 2000),
            })
    except Exception as e:
        print("失败: {} - {}".format(gdir.rgi_id, str(e)[:50]))

print("成功处理: {} 条冰川".format(len(all_anomalies)))

# 转DataFrame
import pandas as pd
df_anom = pd.DataFrame(all_anomalies)

# 统计各时期距平分布
print("\n" + "=" * 70)
print("各时期温度距平统计（相对于1961-1990）")
print("=" * 70)

periods = ['dt_1500', 'dt_1600', 'dt_1700', 'dt_1750', 'dt_1800', 'dt_1850', 'dt_1900', 'dt_2000']
period_names = ['1500', '1600', '1700', '1750', '1800', '1850', '1900', '2000']

print("\n{:<10} {:>10} {:>10} {:>10} {:>10} {:>10}".format(
    "时期", "均值", "中位数", "最小", "最大", "标准差"))
print("-" * 62)

for p, name in zip(periods, period_names):
    vals = df_anom[p].dropna()
    print("{:<10} {:>+10.2f} {:>+10.2f} {:>+10.2f} {:>+10.2f} {:>10.2f}".format(
        name, vals.mean(), vals.median(), vals.min(), vals.max(), vals.std()))

# 绘制箱线图
fig, ax = plt.subplots(figsize=(12, 6))

box_data = [df_anom[p].dropna().values for p in periods]
bp = ax.boxplot(box_data, labels=period_names, patch_artist=True)

for patch in bp['boxes']:
    patch.set_facecolor('lightblue')

ax.axhline(0, color='red', linestyle='--', alpha=0.7, label='参考期(1961-1990)')
ax.set_xlabel('时期')
ax.set_ylabel('温度距平 (°C)')
ax.set_title('210条冰川LMR温度距平分布')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lmr_anomaly_distribution.png', dpi=150)
plt.show()

# 1700→2000升温幅度分布
df_anom['warming_1700_2000'] = df_anom['dt_2000'] - df_anom['dt_1700']

print("\n" + "=" * 70)
print("1700→2000升温幅度分布")
print("=" * 70)
print("均值: {:.2f}°C".format(df_anom['warming_1700_2000'].mean()))
print("中位数: {:.2f}°C".format(df_anom['warming_1700_2000'].median()))
print("范围: {:.2f} ~ {:.2f}°C".format(
    df_anom['warming_1700_2000'].min(), df_anom['warming_1700_2000'].max()))